# Optimización de Modelos Conjunto Soleado por GMM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [3]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [4]:
datos_dia = datos[datos["Cluster GMM"] == "Nublado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
35,2022-09-02 11:00:00,17036.043251,20,0,71,2,2,14,11,Soleado,Nublado,5030.740421,22189.147406
36,2022-09-02 12:00:00,27523.885172,21,0,57,4,2,13,12,Soleado,Nublado,17036.043251,29196.986647
227,2022-09-10 11:00:00,26530.520060,19,0,74,4,1,14,11,Soleado,Nublado,16754.858938,23013.622622
228,2022-09-10 12:00:00,28976.805233,20,0,62,7,1,13,12,Soleado,Nublado,26530.520060,16313.782163
275,2022-09-12 11:00:00,27101.921674,18,0,82,4,2,15,11,Soleado,Nublado,21597.209964,23484.596732
276,2022-09-12 12:00:00,30000.000000,19,0,69,7,2,13,12,Soleado,Nublado,27101.921674,22800.000000
277,2022-09-12 13:00:00,27281.964164,21,0,58,10,2,12,13,Soleado,Nublado,30000.000000,21600.000000
299,2022-09-13 11:00:00,17821.243227,17,0,72,4,2,12,11,Soleado,Nublado,21580.173881,27101.921674
300,2022-09-13 12:00:00,19421.129165,19,0,61,7,2,12,12,Soleado,Nublado,17821.243227,30000.000000
491,2022-09-21 11:00:00,17160.408876,18,0,76,4,2,13,11,Soleado,Nublado,23029.146524,12327.870821


In [5]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [6]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,20,0,71,2,2,14,11,5030.740421,22189.147406
36,21,0,57,4,2,13,12,17036.043251,29196.986647
227,19,0,74,4,1,14,11,16754.858938,23013.622622
228,20,0,62,7,1,13,12,26530.520060,16313.782163
275,18,0,82,4,2,15,11,21597.209964,23484.596732
...,...,...,...,...,...,...,...,...,...
18250,14,0,93,2,1,13,9,7302.000000,17824.000000
18251,17,0,78,4,1,13,10,18014.000000,23097.000000
18252,20,0,64,6,1,13,11,23010.000000,26140.000000
18285,22,0,45,0,1,9,20,1450.000000,0.000000


In [7]:
y = datos_dia[['Generación']]
y

,Generación
35,17036.043251
36,27523.885172
227,26530.520060
228,28976.805233
275,27101.921674
...,...
18250,18014.000000
18251,23010.000000
18252,26156.000000
18285,0.000000


Dividimos entrenamiento, validación y prueba

In [8]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [9]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 1847, y_train: 1847
X_val: 396, y_val: 396
X_test: 396, y_test: 396


## Escalar con MinMaxScaler

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [12]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.52631579 0.         0.69473684 ... 0.33333333 0.16769135 0.73963825]
 [0.55263158 0.         0.54736842 ... 0.4        0.56786811 0.97323289]
 [0.5        0.         0.72631579 ... 0.33333333 0.5584953  0.76712075]
 ...
 [0.44736842 0.         0.25263158 ... 1.         0.         0.        ]
 [0.18421053 0.         0.72631579 ... 0.         0.         0.        ]
 [0.18421053 0.         0.75789474 ... 0.06666667 0.         0.        ]]
(1847, 9)


In [13]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,0.526316,0.0,0.694737,0.153846,0.333333,0.777778,0.333333,0.167691,0.739638
36,0.552632,0.0,0.547368,0.307692,0.333333,0.722222,0.400000,0.567868,0.973233
227,0.500000,0.0,0.726316,0.307692,0.000000,0.777778,0.333333,0.558495,0.767121
228,0.526316,0.0,0.600000,0.538462,0.000000,0.722222,0.400000,0.884351,0.543793
275,0.473684,0.0,0.810526,0.307692,0.333333,0.833333,0.333333,0.719907,0.782820
...,...,...,...,...,...,...,...,...,...
11733,0.421053,0.0,0.473684,0.000000,0.333333,0.277778,0.933333,0.036533,0.000000
11757,0.500000,0.0,0.210526,0.000000,0.666667,0.055556,0.933333,0.030500,0.000000
11758,0.447368,0.0,0.252632,0.000000,0.333333,0.055556,1.000000,0.000000,0.000000
11767,0.184211,0.0,0.726316,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000


In [14]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.13157895 0.         0.84210526 ... 0.13333333 0.         0.00983333]
 [0.18421053 0.         0.78947368 ... 0.2        0.02443333 0.18196667]
 [0.26315789 0.         0.64210526 ... 0.26666667 0.53976667 0.31103333]
 ...
 [0.18421053 0.         0.90526316 ... 0.13333333 0.043      0.34236667]
 [0.23684211 0.         0.85263158 ... 0.2        0.39413333 0.81946667]
 [0.34210526 0.         0.65263158 ... 0.26666667 0.84823333 0.9256    ]]
(396, 9)


In [15]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
11769,0.131579,0.0,0.842105,0.000000,0.0,0.166667,0.133333,0.000000,0.009833
11770,0.184211,0.0,0.789474,0.076923,0.0,0.222222,0.200000,0.024433,0.181967
11771,0.263158,0.0,0.642105,0.153846,0.0,0.222222,0.266667,0.539767,0.311033
11772,0.368421,0.0,0.463158,0.307692,0.0,0.222222,0.333333,0.914833,0.793067
11773,0.473684,0.0,0.326316,0.307692,0.0,0.166667,0.400000,0.929333,0.853733
...,...,...,...,...,...,...,...,...,...
13951,0.210526,0.0,0.852632,0.000000,0.0,0.333333,0.000000,0.000000,0.000000
13952,0.184211,0.0,0.936842,0.000000,0.0,0.333333,0.066667,0.000000,0.013300
13953,0.184211,0.0,0.905263,0.076923,0.0,0.333333,0.133333,0.043000,0.342367
13954,0.236842,0.0,0.852632,0.230769,0.0,0.333333,0.200000,0.394133,0.819467


In [16]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.44736842 0.         0.45263158 ... 0.33333333 0.97296667 0.94383333]
 [0.71052632 0.         0.08421053 ... 0.6        0.98766667 0.93226667]
 [0.76315789 0.         0.05263158 ... 0.66666667 0.96936667 0.91333333]
 ...
 [0.52631579 0.         0.62105263 ... 0.33333333 0.767      0.87133333]
 [0.57894737 0.         0.42105263 ... 0.93333333 0.04833333 0.        ]
 [0.52631579 0.         0.51578947 ... 1.         0.         0.        ]]
(396, 9)


In [17]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
13956,0.447368,0.0,0.452632,0.538462,0.0,0.333333,0.333333,0.972967,0.943833
13960,0.710526,0.0,0.084211,0.538462,0.0,0.166667,0.600000,0.987667,0.932267
13961,0.763158,0.0,0.052632,0.384615,0.0,0.277778,0.666667,0.969367,0.913333
13962,0.815789,0.0,0.042105,0.230769,0.0,0.277778,0.733333,0.955300,0.877900
13963,0.763158,0.0,0.073684,0.076923,0.0,0.166667,0.800000,0.880333,0.708767
...,...,...,...,...,...,...,...,...,...
18250,0.368421,0.0,0.926316,0.153846,0.0,0.722222,0.200000,0.243400,0.594133
18251,0.447368,0.0,0.768421,0.307692,0.0,0.722222,0.266667,0.600467,0.769900
18252,0.526316,0.0,0.621053,0.461538,0.0,0.722222,0.333333,0.767000,0.871333
18285,0.578947,0.0,0.421053,0.000000,0.0,0.500000,0.933333,0.048333,0.000000


In [18]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [19]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.51282051 0.         0.70103093 ... 0.33333333 0.16769135 0.73963825]
 [0.53846154 0.         0.55670103 ... 0.4        0.56786811 0.97323289]
 [0.48717949 0.         0.73195876 ... 0.33333333 0.5584953  0.76712075]
 ...
 [0.51282051 0.         0.62886598 ... 0.33333333 0.767      0.87133333]
 [0.56410256 0.         0.43298969 ... 0.93333333 0.04833333 0.        ]
 [0.51282051 0.         0.5257732  ... 1.         0.         0.        ]]
(2639, 9)


In [20]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,0.512821,0.0,0.701031,0.153846,0.333333,0.777778,0.333333,0.167691,0.739638
36,0.538462,0.0,0.556701,0.307692,0.333333,0.722222,0.400000,0.567868,0.973233
227,0.487179,0.0,0.731959,0.307692,0.000000,0.777778,0.333333,0.558495,0.767121
228,0.512821,0.0,0.608247,0.538462,0.000000,0.722222,0.400000,0.884351,0.543793
275,0.461538,0.0,0.814433,0.307692,0.333333,0.833333,0.333333,0.719907,0.782820
...,...,...,...,...,...,...,...,...,...
18250,0.358974,0.0,0.927835,0.153846,0.000000,0.722222,0.200000,0.243400,0.594133
18251,0.435897,0.0,0.773196,0.307692,0.000000,0.722222,0.266667,0.600467,0.769900
18252,0.512821,0.0,0.628866,0.461538,0.000000,0.722222,0.333333,0.767000,0.871333
18285,0.564103,0.0,0.432990,0.000000,0.000000,0.500000,0.933333,0.048333,0.000000


In [21]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [22]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.56786811]
 [0.91746284]
 [0.88435067]
 ...
 [0.        ]
 [0.        ]
 [0.        ]]
(1847, 1)


In [23]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
35,0.567868
36,0.917463
227,0.884351
228,0.965894
275,0.903397
...,...
11733,0.000000
11757,0.000000
11758,0.000000
11767,0.000000


In [24]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[2.44333333e-02]
 [5.39766667e-01]
 [9.14833333e-01]
 [9.29333333e-01]
 [9.46266667e-01]
 [8.36400000e-01]
 [8.03333333e-01]
 [8.37600000e-01]
 [8.36633333e-01]
 [8.09166667e-01]
 [8.36766667e-01]
 [8.12266667e-01]
 [4.06900000e-01]
 [3.18666667e-02]
 [8.06933333e-01]
 [8.85000000e-01]
 [8.08366667e-01]
 [3.55366667e-01]
 [8.36400000e-01]
 [8.50366667e-01]
 [8.51533333e-01]
 [8.37533333e-01]
 [8.05866667e-01]
 [8.36866667e-01]
 [8.09666667e-01]
 [4.06400000e-01]
 [3.15333333e-02]
 [3.15333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [9.34133333e-01]
 [9.49200000e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.45333333e-02]
 [5.35233333e-01]
 [9.49400000e-01]
 [9.30866667e-01]
 [9.48733333e-01]
 [9.43500000e-01]
 [9.33433333e-01]
 [8.93566667e-01]
 [9.29733333e-01]
 [8.98200000e-01]
 [4.51566667e-01]
 [3.57666667e-02]
 [0.00000000e+00]
 [8.30000000e-03]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.54233333e-01]
 [7.31700000e-01]
 [7.93966667e-01]
 [0.000000

In [25]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
11769,0.024433
11770,0.539767
11771,0.914833
11772,0.929333
11773,0.946267
...,...
13951,0.000000
13952,0.043000
13953,0.394133
13954,0.848233


In [26]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[9.95466667e-01]
 [9.69366667e-01]
 [9.55300000e-01]
 [8.80333333e-01]
 [5.72166667e-01]
 [7.90133333e-01]
 [0.00000000e+00]
 [1.43000000e-02]
 [2.74900000e-01]
 [6.55566667e-01]
 [1.53333333e-03]
 [2.79866667e-01]
 [6.67100000e-01]
 [7.40466667e-01]
 [6.19600000e-01]
 [5.03100000e-01]
 [1.34666667e-02]
 [3.42366667e-01]
 [8.34100000e-01]
 [9.25600000e-01]
 [9.35966667e-01]
 [9.50600000e-01]
 [9.28233333e-01]
 [9.34933333e-01]
 [9.27933333e-01]
 [0.00000000e+00]
 [1.81666667e-02]
 [3.80700000e-01]
 [8.66833333e-01]
 [9.74500000e-01]
 [9.95466667e-01]
 [9.95433333e-01]
 [9.94366667e-01]
 [9.85700000e-01]
 [9.95466667e-01]
 [9.61066667e-01]
 [8.29166667e-01]
 [5.44333333e-01]
 [1.08133333e-01]
 [3.33333333e-05]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.35666667e-02]
 [3.43266667e-01]
 [8.19466667e-01]
 [9.25600000e-01]
 [0.00000000e+00]
 [6.71400000e-01]
 [7.48566667e-01]
 [8.18700000e-01]
 [0.00000000e+00]
 [1.19666667e-02]
 [3.10700000e-01]
 [7.17966667e-01]
 [7.62200000e-01]
 [7.969000

In [27]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
13956,0.995467
13960,0.969367
13961,0.955300
13962,0.880333
13963,0.572167
...,...
18250,0.600467
18251,0.767000
18252,0.871867
18285,0.000000


In [28]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [29]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.56786811]
 [0.91746284]
 [0.88435067]
 ...
 [0.87186667]
 [0.        ]
 [0.        ]]
(2639, 1)


In [30]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
35,0.567868
36,0.917463
227,0.884351
228,0.965894
275,0.903397
...,...
18250,0.600467
18251,0.767000
18252,0.871867
18285,0.000000


## Preparación para Redes Neuronales

In [31]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [32]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [33]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (1799, 48, 9), y_train: (1799, 1)
X_val: (348, 48, 9), y_val: (348, 1)
X_test: (348, 48, 9), y_test: (348, 1)


## Optuna

In [34]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.4 MB/s eta 0:00:00


In [35]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [36]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-13 19:11:17,523] A new study created in memory with name: no-name-895b1852-5469-4045-8399-da8d092f51c0
[I 2025-03-13 19:11:17,628] Trial 0 finished with value: 0.014225924829027676 and parameters: {'num_leaves': 366, 'subsample': 0.8420166063387758, 'colsample_bytree': 0.9938548688928597, 'min_data_in_leaf': 83}. Best is trial 0 with value: 0.014225924829027676.


[LightGBM] [Warning] min_data_in_leaf is set=83, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=83
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=83, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=83
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001018 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 696
[LightGBM] [Info] Number of data points in the train set: 1847, number of used features: 8
[LightGBM] [Info] Start training from score 0.454230
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

[I 2025-03-13 19:11:17,755] Trial 1 finished with value: 0.015491058596329772 and parameters: {'num_leaves': 953, 'subsample': 0.9828438485517351, 'colsample_bytree': 0.5938080478911444, 'min_data_in_leaf': 20}. Best is trial 0 with value: 0.014225924829027676.
[I 2025-03-13 19:11:17,823] Trial 2 finished with value: 0.016417882855020906 and parameters: {'num_leaves': 57, 'subsample': 0.37247322279233575, 'colsample_bytree': 0.5377981154356072, 'min_data_in_leaf': 30}. Best is trial 0 with value: 0.014225924829027676.
[I 2025-03-13 19:11:17,858] Trial 3 finished with value: 0.013620358078733302 and parameters: {'num_leaves': 388, 'subsample': 0.32374566828935325, 'colsample_bytree': 0.36832350225886734, 'min_data_in_leaf': 91}. Best is trial 3 with value: 0.013620358078733302.


[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000101 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 696
[LightGBM] [Info] Number of data points in the train set: 1847, number of used features: 8
[LightGBM] [Info] Start training from score 0.454230
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

[I 2025-03-13 19:11:18,015] Trial 4 finished with value: 0.015265851344502859 and parameters: {'num_leaves': 482, 'subsample': 0.5176026914373232, 'colsample_bytree': 0.4075253527569206, 'min_data_in_leaf': 12}. Best is trial 3 with value: 0.013620358078733302.
[I 2025-03-13 19:11:18,077] Trial 5 finished with value: 0.014909603175605256 and parameters: {'num_leaves': 644, 'subsample': 0.7755163243015978, 'colsample_bytree': 0.7813268530326989, 'min_data_in_leaf': 63}. Best is trial 3 with value: 0.013620358078733302.
[I 2025-03-13 19:11:18,130] Trial 6 finished with value: 0.013632038629276602 and parameters: {'num_leaves': 201, 'subsample': 0.5121636063238033, 'colsample_bytree': 0.5952924583810109, 'min_data_in_leaf': 68}. Best is trial 3 with value: 0.013620358078733302.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-13 19:11:20,632] Trial 45 finished with value: 0.012706608826790358 and parameters: {'num_leaves': 163, 'subsample': 0.4978162913983965, 'colsample_bytree': 0.7571756011882436, 'min_data_in_leaf': 91}. Best is trial 23 with value: 0.01219272713521226.
[I 2025-03-13 19:11:20,703] Trial 46 finished with value: 0.014097738075205857 and parameters: {'num_leaves': 636, 'subsample': 0.577109333041042, 'colsample_bytree': 0.8374338920605728, 'min_data_in_leaf': 84}. Best is trial 23 with value: 0.01219272713521226.
[I 2025-03-13 19:11:20,766] Trial 47 finished with value: 0.012797685883319124 and parameters: {'num_leaves': 711, 'subsample': 0.673470770583751, 'colsample_bytree': 0.9176530540436025, 'min_data_in_leaf': 96}. Best is trial 23 with value: 0.01219272713521226.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-13 19:11:20,835] Trial 48 finished with value: 0.014140572492935051 and parameters: {'num_leaves': 425, 'subsample': 0.7757143923650247, 'colsample_bytree': 0.8631098467878671, 'min_data_in_leaf': 79}. Best is trial 23 with value: 0.01219272713521226.
[I 2025-03-13 19:11:20,903] Trial 49 finished with value: 0.013271416978703536 and parameters: {'num_leaves': 357, 'subsample': 0.6115237986743339, 'colsample_bytree': 0.7252333751629324, 'min_data_in_leaf': 92}. Best is trial 23 with value: 0.01219272713521226.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

### Random Forest

In [37]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-13 19:11:20,911] A new study created in memory with name: no-name-365457a7-3ab7-4c53-9560-5b8501416ab9
[I 2025-03-13 19:11:23,613] Trial 0 finished with value: 0.023546420316055156 and parameters: {'n_estimators': 400, 'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 8, 'bootstrap': False}. Best is trial 0 with value: 0.023546420316055156.
[I 2025-03-13 19:11:24,764] Trial 1 finished with value: 0.025916091413395084 and parameters: {'n_estimators': 150, 'max_depth': 25, 'min_samples_split': 12, 'min_samples_leaf': 3, 'bootstrap': False}. Best is trial 0 with value: 0.023546420316055156.
[I 2025-03-13 19:11:28,021] Trial 2 finished with value: 0.024524993668629164 and parameters: {'n_estimators': 500, 'max_depth': 40, 'min_samples_split': 9, 'min_samples_leaf': 9, 'bootstrap': False}. Best is trial 0 with value: 0.023546420316055156.
[I 2025-03-13 19:11:29,888] Trial 3 finished with value: 0.02044489736031323 and parameters: {'n_estimators': 400, 'max_depth': 20,

Mejores hiperparámetros: {'n_estimators': 250, 'max_depth': 15, 'min_samples_split': 4, 'min_samples_leaf': 10, 'bootstrap': True}


### CTNET

In [38]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [39]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [40]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-13 19:12:39,112] A new study created in memory with name: no-name-910199f4-3121-4a3e-82fd-b7d0712aa246
[I 2025-03-13 19:13:44,384] Trial 0 finished with value: 0.11204507201910019 and parameters: {'head_size': 2, 'num_heads': 3, 'ff_dim': 128, 'num_transformer_blocks': 3, 'mlp_units_1': 64, 'mlp_units_2': 32, 'dropout': 0.2465931976918701, 'mlp_dropout': 0.3258609909448631, 'learning_rate': 7.86813855634637e-05, 'batch_size': 512}. Best is trial 0 with value: 0.11204507201910019.
[I 2025-03-13 19:14:24,631] Trial 1 finished with value: 0.15663477778434753 and parameters: {'head_size': 6, 'num_heads': 4, 'ff_dim': 128, 'num_transformer_blocks': 1, 'mlp_units_1': 192, 'mlp_units_2': 192, 'dropout': 0.3846429273391544, 'mlp_dropout': 0.22505376506310976, 'learning_rate': 4.2573461860700096e-05, 'batch_size': 256}. Best is trial 0 with value: 0.11204507201910019.
[I 2025-03-13 19:15:16,503] Trial 2 finished with value: 0.15723513066768646 and parameters: {'head_size': 8, 'num_he

Mejores hiperparámetros: {'head_size': 7, 'num_heads': 6, 'ff_dim': 112, 'num_transformer_blocks': 4, 'mlp_units_1': 256, 'mlp_units_2': 160, 'dropout': 0.20184163593744978, 'mlp_dropout': 0.16747221003829596, 'learning_rate': 0.0012189326492547325, 'batch_size': 256}


### Forescasting

In [43]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-13 20:26:03,713] A new study created in memory with name: no-name-8d64fee2-3ab2-441f-aca1-98ab8c20cc2a
[I 2025-03-13 20:27:27,462] Trial 9 finished with value: 0.45595037937164307 and parameters: {'filters': 32, 'kernel_size': 5, 'lstm_units_1': 64, 'lstm_units_2': 32, 'lstm_units_3': 16, 'dropout_lstm': 0.2742956039740999, 'dropout_dense': 0.4753189390756696, 'learning_rate': 0.00180749666818211, 'batch_size': 128}. Best is trial 9 with value: 0.45595037937164307.
[I 2025-03-13 20:27:39,241] Trial 12 finished with value: 0.4559503197669983 and parameters: {'filters': 32, 'kernel_size': 3, 'lstm_units_1': 256, 'lstm_units_2': 128, 'lstm_units_3': 64, 'dropout_lstm': 0.22536444320378118, 'dropout_dense': 0.3201371527365062, 'learning_rate': 0.008854074136585016, 'batch_size': 512}. Best is trial 12 with value: 0.4559503197669983.
[I 2025-03-13 20:27:50,833] Trial 13 finished with value: 0.37292414903640747 and parameters: {'filters': 32, 'kernel_size': 4, 'lstm_units_1': 128,

Mejores hiperparámetros: {'filters': 128, 'kernel_size': 5, 'lstm_units_1': 64, 'lstm_units_2': 32, 'lstm_units_3': 32, 'dropout_lstm': 0.40041063539523436, 'dropout_dense': 0.1716619465424395, 'learning_rate': 0.001445396577181948, 'batch_size': 128}


### Photovoltaic

In [44]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-13 20:36:25,115] A new study created in memory with name: no-name-64dde98e-3ebf-409f-80d5-fe0ffa914115


Epoch 40: early stopping
Restoring model weights from the end of the best epoch: 30.


[I 2025-03-13 20:37:48,678] Trial 4 finished with value: 0.07642974704504013 and parameters: {'filters_1': 64, 'filters_2': 256, 'kernel_size': 5, 'gru_units': 32, 'num_heads': 8, 'dropout_rate': 0.2559022617533918, 'dense_units_1': 64, 'dense_units_2': 32, 'learning_rate': 0.003362127120707278, 'batch_size': 256}. Best is trial 4 with value: 0.07642974704504013.


Epoch 52: early stopping
Restoring model weights from the end of the best epoch: 42.


[I 2025-03-13 20:37:57,963] Trial 9 finished with value: 0.07989092171192169 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 3, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.26972583634499225, 'dense_units_1': 32, 'dense_units_2': 16, 'learning_rate': 0.0006692418097709164, 'batch_size': 256}. Best is trial 4 with value: 0.07642974704504013.


...

Epoch 66: early stopping
Restoring model weights from the end of the best epoch: 56.


[I 2025-03-13 20:43:00,567] Trial 45 finished with value: 0.08176463842391968 and parameters: {'filters_1': 64, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 128, 'num_heads': 2, 'dropout_rate': 0.2903432954835818, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.002354006087143089, 'batch_size': 256}. Best is trial 39 with value: 0.06859374046325684.


Epoch 78: early stopping
Restoring model weights from the end of the best epoch: 68.


[I 2025-03-13 20:43:03,707] Trial 47 finished with value: 0.07173342257738113 and parameters: {'filters_1': 64, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 128, 'num_heads': 2, 'dropout_rate': 0.28123892284159113, 'dense_units_1': 64, 'dense_units_2': 16, 'learning_rate': 0.00262956202297221, 'batch_size': 256}. Best is trial 39 with value: 0.06859374046325684.


Mejores hiperparámetros: {'filters_1': 64, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 128, 'num_heads': 2, 'dropout_rate': 0.2976485622510353, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.0009167604110354238, 'batch_size': 256}
